Author: Diane Menuz  
Date: 6/11/2026  

Goal: Estimate crop height for a corn field, using 2025 data from Green River as an  
example

General process
- Determine crop planting and cut dates. Crop planting date in this example is taken as the   
date that the standing dead stems were cut down in the spring. Both dates were taken by  
examining timelapse images.
- Input field height data into the field_heights_dict, in centimeters
- Select k rate and other model parameters during the active growth period.
- Fill in height data before and after the crop planting dates and when the crop growth stabilizes
- Format and export data


In [2]:
import pandas as pd
import sys
sys.path.append('../../../src')
from micromet import eddy_plots as ed_plot
from micromet import simulate_alfalfa_height_single_field, AlfalfaHeightParams

Field-measured crop height measurements, in centimeters. Data should be examined so that any  
outliers are removed.

In [3]:
field_heights_dict = {
    '2025-06-17':	42.7,
    '2025-07-17':	152.4,
    '2025-08-12': 237.7,
    '2025-08-15': 	237.7,
    '2025-10-03':274.3,
    '2025-10-24': 280.4
}

field_height = pd.Series(field_heights_dict, name='height_cm').to_frame()       
field_height.index = pd.to_datetime(field_height.index)

Run model, testing different parameters
- growth_start_date and growth_end_date are the range of dates over which growth will be  
simulated. growth_start_date in this example was selected as the date after the stubble  
from previous years was first cut down. growth_end_date was selected as the date when the  
growth was expected to stabilize to a set height (since the corn doesn't grow indefinitely).  
You can determine this latter date by looking at field measurements or estimating early fall.
- max_height: maximum estimated height; height at which corn growth will level out
- k_rate: growth rate for the logistic model

In [4]:
growth_start_date = '2025-05-25'
growth_end_date = '2025-10-04'
max_height = 276
k_rate = 0.062

params = AlfalfaHeightParams(
        h_resid_cm=12, 
        h_max_cm=max_height,
        rate=k_rate,       
        model="logistic",
        time_mode="doy",
        dormancy_mode="none",
        drawdown=False
    )

growth_curve = simulate_alfalfa_height_single_field(
        dates= pd.date_range(growth_start_date, growth_end_date, freq="D"),
        cut_dates = [],
        params = params,
        weather = None,
        cut_effect= "post")

ed_plot.plotlystuff([growth_curve, field_height],
                    ['height_cm','height_cm'],
                    ['lines', 'markers'],
                    plot_height = 300)

Once the growth curve fits the data points pretty well, fill out the rest of the  
year of data.
- fall_cut_date: Date when crop is cut back to stubble
- cut_height: stubble height in cm both before growth and after cut date
- max_height parameter is pulled from above

In [5]:
fall_cut_date = '2025-12-21'
cut_height = 107

year = growth_curve.index.year[0]
full_calendar = pd.date_range(
    start=pd.to_datetime(f'{year}-01-01'), 
    end= pd.to_datetime(f'{year}-12-31'), 
    freq="D")
daily_height = growth_curve.reindex(full_calendar)
mask = (daily_height.index>pd.to_datetime(growth_end_date)) & (
    daily_height.index<pd.to_datetime(fall_cut_date))
daily_height.loc[mask, 'height_cm'] = max_height
daily_height['height_cm'] = daily_height['height_cm'].fillna(cut_height)

ed_plot.plotlystuff([daily_height, field_height],
                    ['height_cm','height_cm'],
                    ['lines', 'markers'],
                    plot_height = 300)

In [6]:
stationid = "US-UTG"
zone = 'W'

final_height = daily_height.copy()
final_height['stationid'] = stationid
final_height['height_m'] = final_height['height_cm']/100
final_height['height_m'] = final_height['height_m'].round(2)
final_height['zone_name'] = zone
final_height['measurement_type'] = 'log doy model'
final_height.reset_index(inplace=True)
final_height.rename(columns={'index':'height_date'}, inplace=True)
final_height.drop(columns=['active_k', 'height_cm'], inplace=True)
final_height.head(2)

,height_date,stationid,height_m,zone_name,measurement_type
0,2025-01-01,US-UTG,1.07,W,log doy model
1,2025-01-02,US-UTG,1.07,W,log doy model


Now you can export the data or import into a database